# 법안 분야 자동 분류 (C조 입법 영향 지도 「빼곡」)

국회 법안의 제안이유 글을 읽고 9개 분야 중 하나로 나누는 신경망을 직접 학습한다.
9주차 숫자 인식(MNIST) 실습의 Dense 신경망을 글 분류에 그대로 옮겨 쓴다.

**순서**: 1 문제 → 2 데이터 → 3 전처리 → 4 학습/테스트 분리 → 5 기준선 → 6 신경망 → 7 과적합 확인 → 8 은닉층 실험 → 9 오답 분석 → 10 결과표 → 11 화면에 쓰이는 곳

## 1. 문제

**무엇을**: 법안 1건의 제안이유 글을 넣으면 9개 분야 중 어느 분야인지 고른다.

**왜**: 지도 화면과 관리자 화면에서 법안을 분야로 거르려면 법안마다 분야가 있어야 한다. 법안은 계속 늘어나서(지금 약 700건) 사람이 모두 붙이기 어렵다. 그래서 사람이 300건에 분야를 붙이고, 나머지는 이 모델이 붙인다.

- 입력: 제안이유 글 (머리말 "제안이유 및 주요내용"을 뗀 본문)
- 출력: 9개 분야 각각의 확률 (합 = 1). 가장 큰 것을 분야로, 그 확률을 확신도로 쓴다.

| 분야 번호 | 분야 |
|---|---|
| 1 | 소방·화재 |
| 2 | 재난·안전관리 |
| 3 | 건축물 안전·노후·유지관리 |
| 4 | 주택 공급·임대·금융 |
| 5 | 공동주택 관리·주거환경 |
| 6 | 정비사업·도시재생 |
| 7 | 도시계획·건축규제 |
| 8 | 주차·보행·교통 |
| 9 | 건설산업·기술·기타 |

**9주차 숫자 인식과 비교**

| 9주차 숫자 인식 | 이 노트북 |
|---|---|
| 손글씨 이미지 28×28 = 784픽셀 | 법안 제안이유 → TF-IDF 벡터 3,000차원 |
| 정답 0~9 (10개) | 분야 1~9 (9개) |
| Dense 512 → 256 → Softmax 10 | Dense 512 → 256 → Softmax 9 (8절에서 더 작은 구조와 비교) |
| 학습·테스트 정확도, 오답 분석 | 같음 + 규칙 기반·로지스틱 회귀 기준선과 비교 |

## 2. 데이터

- `labels_300.csv`: 사람이 분야를 붙인 법안 300건 (LABEL 1~9, 학습 제외 표시 EXCLUDE, 애매 표시 AMBIG)
- `bills_summary.csv`: 수집한 법안 전체와 제안이유 (예측 대상)

GitHub 저장소 main 의 `data/` 에서 읽는다. 인터넷이 안 되면 `NB_LOCAL_DATA` 폴더, 그다음 `../data/` 를 읽고, 그래도 없으면 (Colab) 파일 올리기 창을 띄운다.

In [ ]:
import os, sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'          # TensorFlow 안내 메시지 숨기기 (tf 를 불러오기 전에)
if 'google.colab' in sys.modules and not os.path.exists('/usr/share/fonts/truetype/nanum'):
    !apt-get -qq install fonts-nanum > /dev/null

import re, json, base64, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
tf.get_logger().setLevel('ERROR')
warnings.filterwarnings('ignore')

import matplotlib.font_manager as fm
_f = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(_f):
    fm.fontManager.addfont(_f)                       # 세션 다시 시작 없이 한글 글꼴 등록
    plt.rcParams['font.family'] = fm.FontProperties(fname=_f).get_name()
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()   # 다시 실행해도 같은 결과

FIELD = {1:'소방·화재', 2:'재난·안전관리', 3:'건축물 안전·노후·유지관리', 4:'주택 공급·임대·금융',
         5:'공동주택 관리·주거환경', 6:'정비사업·도시재생', 7:'도시계획·건축규제', 8:'주차·보행·교통', 9:'건설산업·기술·기타'}
print('TensorFlow', tf.__version__)

In [ ]:
BASE = 'https://raw.githubusercontent.com/cbnu-lawmap/law-impact-map/main/data/'
LOCAL_DIRS = [d for d in [os.environ.get('NB_LOCAL_DATA'), '../data', '.'] if d]

def load(name, required=True):
    try:
        d = pd.read_csv(BASE + name, encoding='utf-8-sig', dtype={'BILL_NO': str})
        print(name, '← GitHub main')
        return d
    except Exception:
        pass
    for folder in LOCAL_DIRS:
        p = os.path.join(folder, name)
        if os.path.exists(p):
            print(name, '←', p)
            return pd.read_csv(p, encoding='utf-8-sig', dtype={'BILL_NO': str})
    if not required:
        return None
    from google.colab import files
    print(name, '파일을 올려 주세요'); files.upload()
    return pd.read_csv(name, encoding='utf-8-sig', dtype={'BILL_NO': str})

lab = load('labels_300.csv')        # 라벨 300건
allb = load('bills_summary.csv')    # 예측 대상 법안 전체
print('라벨', lab.shape, '/ 전체 법안', allb.shape)
lab[['BILL_NO', 'BILL_NAME', 'LABEL', 'LABEL_NAME', 'EXCLUDE', 'AMBIG']].head(3)

## 3. 전처리

- 학습 제외 표시(EXCLUDE = Y, 용어만 고치는 법안) 빼기
- 제안이유 앞의 머리말("제안이유 및 주요내용")을 떼고 공백 정리
- 법률명 뽑기: 규칙 기준선과 "법률명이 같아도 분야가 갈리는 법률" 구간 분석에 쓴다

In [ ]:
def clean(t):
    t = str(t)
    t = re.sub(r'^\s*제안이유\s*(및\s*주요내용)?', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

def law_name(n):
    n = re.sub(r'\s*(일부|전부)개정법률안.*|\s*법률안.*', '', str(n))
    return re.sub(r'\(.*?\)', '', n).strip()

df = lab[lab['EXCLUDE'] == 'N'].copy()
df['TEXT'] = df['SUMMARY'].map(clean)
df['LAW'] = df['BILL_NAME'].map(law_name)
print('학습에 쓰는 법안:', len(df), '건 (제외', (lab['EXCLUDE'] != 'N').sum(), '건)')
print('제안이유 길이(글자): 중앙값', int(df['TEXT'].str.len().median()), '/ 최대', df['TEXT'].str.len().max())

mixed = df.groupby('LAW')['LABEL'].nunique()
mixed_laws = list(mixed[mixed > 1].index)
print('법률명이 같아도 분야가 갈리는 법률', len(mixed_laws), '개:', mixed_laws)

cnt = df['LABEL'].value_counts().sort_index()
ax = cnt.rename(FIELD).plot.barh(figsize=(7, 3.5))
ax.invert_yaxis(); ax.set_xlabel('법안 수'); ax.set_title('분야별 라벨 수 (학습에 쓰는 법안)')
plt.tight_layout(); plt.show()

## 4. 학습/테스트 분리

8:2 로 나누고 분야 비율은 양쪽이 같게 한다(stratify). 테스트 60건은 학습에 전혀 쓰지 않는다.

글을 숫자로 바꾸는 TF-IDF 도 **훈련 데이터로만** 맞춘다. 이미지는 픽셀 밝기가 숫자였다. 글은 글자 2~3개 묶음(n-gram)이 얼마나 자주, 얼마나 드물게 나오는지를 숫자로 쓴다. 한국어는 띄어쓰기 단위가 길고 조사가 붙어서 낱말보다 글자 묶음이 낫다. 자주 나오는 묶음 3,000개를 골라 법안 1건 = 숫자 3,000개.

In [ ]:
tr, te = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['LABEL'])
tr, te = tr.copy(), te.copy()
print('훈련', len(tr), '/ 테스트', len(te))

def make_vec():
    return TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), max_features=3000, sublinear_tf=True)

vec = make_vec()
X_tr = vec.fit_transform(tr['TEXT']).toarray().astype('float32')
X_te = vec.transform(te['TEXT']).toarray().astype('float32')
y_tr = to_categorical(tr['LABEL'] - 1, num_classes=9)     # 분야 1~9 → 0~8 자리 원-핫
y_te = to_categorical(te['LABEL'] - 1, num_classes=9)
print('입력', X_tr.shape, '/ 정답', y_tr.shape)
print('글자 묶음 예:', list(vec.vocabulary_)[:8])

## 5. 기준선

신경망이 정말 도움이 되는지 보려면 더 단순한 방법과 비교해야 한다. 두 가지를 둔다.

1. **규칙 기반** (1주차: 규칙 기반 AI): "이 법률의 법안은 대부분 이 분야다". 훈련 데이터에서 법률별로 가장 많은 분야를 규칙으로 만들고, 처음 보는 법률은 가장 흔한 분야로 답한다.
2. **로지스틱 회귀**: 신경망과 같은 TF-IDF 입력에 은닉층 없이 바로 Softmax 를 붙인 것과 같다. 규제 세기 C 는 훈련 데이터 안에서 5겹 교차검증으로 고른다(테스트 데이터는 보지 않음).

In [ ]:
def rule_fit(d):
    return d.groupby('LAW')['LABEL'].agg(lambda s: s.value_counts().idxmax()).to_dict(), d['LABEL'].value_counts().idxmax()

def rule_predict(r, d):
    table, default = r
    return d['LAW'].map(table).fillna(default).astype(int).values

rule = rule_fit(tr)
te['RULE'] = rule_predict(rule, te)
acc_rule = accuracy_score(te['LABEL'], te['RULE'])
print('규칙 기반 테스트 정확도: %.3f' % acc_rule)

In [ ]:
gs = GridSearchCV(LogisticRegression(max_iter=5000), {'C': [0.1, 1, 10, 100]},
                  cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
gs.fit(X_tr, tr['LABEL'])
LR_C = gs.best_params_['C']
print('교차검증 정확도:', ', '.join('C=%g: %.3f' % (c, s) for c, s in zip(gs.cv_results_['param_C'], gs.cv_results_['mean_test_score'])))
print('고른 C =', LR_C)

logreg = LogisticRegression(C=LR_C, max_iter=5000).fit(X_tr, tr['LABEL'])
te['LOGREG'] = logreg.predict(X_te)
acc_lr_train = accuracy_score(tr['LABEL'], logreg.predict(X_tr))
acc_lr = accuracy_score(te['LABEL'], te['LOGREG'])
print('로지스틱 회귀  학습 정확도 %.3f / 테스트 정확도 %.3f' % (acc_lr_train, acc_lr))

## 6. 신경망

9주차와 같은 구조: Dense 512 (ReLU) → Dense 256 (ReLU) → Dense 9 (Softmax). 손실은 categorical crossentropy, 최적화는 Adam, 20 epoch, 배치 16. 훈련 데이터의 마지막 10% 는 검증용으로 떼어 학습 곡선을 본다.

In [ ]:
def build(hidden=(512, 256), seed=SEED, n_in=None):
    tf.keras.utils.set_random_seed(seed)
    m = Sequential([Input(shape=(n_in or X_tr.shape[1],))])
    for h in hidden:
        m.add(Dense(h, activation='relu'))
    m.add(Dense(9, activation='softmax'))
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

def proba(m, X):
    return m(X, training=False).numpy()          # predict() 보다 빠르고 반복해도 경고가 없다

EPOCHS = 20
model = build()
model.summary()
hist = model.fit(X_tr, y_tr, epochs=EPOCHS, batch_size=16, validation_split=0.1, verbose=0)
te['NN512'] = proba(model, X_te).argmax(1) + 1
acc_train = accuracy_score(tr['LABEL'], proba(model, X_tr).argmax(1) + 1)
acc_nn = accuracy_score(te['LABEL'], te['NN512'])
print('마지막 epoch  학습 %.3f / 검증 %.3f' % (hist.history['accuracy'][-1], hist.history['val_accuracy'][-1]))
print('신경망(512, 256)  테스트 정확도 %.3f' % acc_nn)

## 7. 과적합 확인

학습 데이터를 거의 외우면(학습 정확도 ≈ 1) 처음 보는 법안에서 정확도가 떨어질 수 있다. 세 가지로 본다.

1. 학습 정확도와 테스트 정확도의 차이
2. 학습 곡선: epoch 이 늘 때 검증 손실이 다시 오르는지
3. 조기 종료(EarlyStopping): 검증 손실이 3 epoch 동안 나아지지 않으면 멈추고 가장 좋았던 가중치로 돌아간다. 20 epoch 고정과 테스트 정확도를 비교한다.

In [ ]:
print('신경망(512, 256)  학습 정확도 %.3f / 테스트 정확도 %.3f / 차이 %.3f' % (acc_train, acc_nn, acc_train - acc_nn))
print('로지스틱 회귀      학습 정확도 %.3f / 테스트 정확도 %.3f / 차이 %.3f' % (acc_lr_train, acc_lr, acc_lr_train - acc_lr))

long_model = build()
long_hist = long_model.fit(X_tr, y_tr, epochs=40, batch_size=16, validation_split=0.1, verbose=0).history

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ep = range(1, 41)
ax[0].plot(ep, long_hist['accuracy'], label='학습'); ax[0].plot(ep, long_hist['val_accuracy'], label='검증')
ax[0].set_title('정확도'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(ep, long_hist['loss'], label='학습'); ax[1].plot(ep, long_hist['val_loss'], label='검증')
ax[1].axvline(EPOCHS, color='gray', ls=':', label='20 epoch')
ax[1].set_title('손실'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.suptitle('학습 곡선 (512, 256), 40 epoch'); plt.tight_layout(); plt.show()
best_ep = int(np.argmin(long_hist['val_loss'])) + 1
print('검증 손실이 가장 낮은 epoch:', best_ep, '/ 그때 검증 정확도 %.3f' % long_hist['val_accuracy'][best_ep - 1])

In [ ]:
es_model = build()
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
es_hist = es_model.fit(X_tr, y_tr, epochs=40, batch_size=16, validation_split=0.1, verbose=0, callbacks=[es])
acc_es = accuracy_score(te['LABEL'], proba(es_model, X_te).argmax(1) + 1)
pd.DataFrame([
    {'학습 방법': '20 epoch 고정', '멈춘 epoch': EPOCHS, '테스트 정확도': round(acc_nn, 3)},
    {'학습 방법': '조기 종료 (patience 3)', '멈춘 epoch': len(es_hist.history['loss']), '테스트 정확도': round(acc_es, 3)},
])

## 8. 은닉층 실험

(4주차: 신경망 구조 설계) 은닉층 구성만 바꿔서 비교한다. 테스트 60건 한 번으로는 1~2건 차이로 숫자가 크게 흔들리므로, **훈련/테스트 분할을 10번 새로 나눠** 같은 비교를 반복하고 평균과 표준편차를 본다. 기준선(규칙 기반, 로지스틱 회귀)도 같은 10번의 분할에서 함께 잰다.

최종 구조 고르는 기준: 평균 정확도가 가장 좋은 구조와 0.01 이내로 비슷하면 **더 작은 구조**를 고른다. 서버가 가중치 파일을 읽어 직접 계산하므로 작을수록 파일이 가볍고 빠르며, 데이터가 적을 때 과적합도 덜하다.

In [ ]:
HIDDENS = [(128,), (256, 128), (512, 256), (512, 256, 128)]
N_SPLIT = 10
rows = []
for r in range(N_SPLIT):
    a, b = train_test_split(df, test_size=0.2, random_state=100 + r, stratify=df['LABEL'])
    v = make_vec()
    xa = v.fit_transform(a['TEXT']).toarray().astype('float32'); xb = v.transform(b['TEXT']).toarray().astype('float32')
    sg = b['LAW'].isin(mixed_laws).values
    preds = {'규칙 기반': rule_predict(rule_fit(a), b),
             '로지스틱 회귀': LogisticRegression(C=LR_C, max_iter=5000).fit(xa, a['LABEL']).predict(xb)}
    for h in HIDDENS:
        m = build(h, seed=SEED + r, n_in=xa.shape[1])
        m.fit(xa, to_categorical(a['LABEL'] - 1, 9), epochs=EPOCHS, batch_size=16, verbose=0)
        preds['신경망 ' + str(h)] = proba(m, xb).argmax(1) + 1
    y = b['LABEL'].values
    for name, p in preds.items():
        rows.append({'분할': r + 1, '모델': name, '전체': accuracy_score(y, p),
                     '분야가 갈리는 법률': accuracy_score(y[sg], p[sg]), '분야가 하나인 법률': accuracy_score(y[~sg], p[~sg])})
rep = pd.DataFrame(rows)
order = ['규칙 기반', '로지스틱 회귀'] + ['신경망 ' + str(h) for h in HIDDENS]
summary = rep.groupby('모델')[['전체', '분야가 갈리는 법률', '분야가 하나인 법률']].agg(['mean', 'std']).reindex(order).round(3)
summary

In [ ]:
cv = rep.groupby('모델')['전체'].agg(['mean', 'std']).reindex(order)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(range(len(cv)), cv['mean'], yerr=cv['std'], capsize=4, color=['#999', '#999'] + ['#4a7ab5'] * len(HIDDENS))
ax.set_xticks(range(len(cv))); ax.set_xticklabels(cv.index, fontsize=9)
ax.set_ylim(0.5, 1); ax.set_ylabel('테스트 정확도'); ax.set_title('분할 10번 평균 (막대 위 선 = 표준편차)')
for i, (mu, sd) in enumerate(zip(cv['mean'], cv['std'])):
    ax.text(i, mu + sd + 0.01, '%.3f' % mu, ha='center', fontsize=9)
plt.tight_layout(); plt.show()

nn_cv = cv.loc[['신경망 ' + str(h) for h in HIDDENS], 'mean']
best = nn_cv.max()
FINAL_HIDDEN = next(h for h in HIDDENS if nn_cv['신경망 ' + str(h)] >= best - 0.01)   # HIDDENS 는 작은 것부터
n_params = lambda h: sum(i * o + o for i, o in zip((3000,) + h, h + (9,)))
print('가장 좋은 평균 %.3f / 최종 구조 %s (평균 %.3f, 가중치 %s개)' % (best, FINAL_HIDDEN, nn_cv['신경망 ' + str(FINAL_HIDDEN)], format(n_params(FINAL_HIDDEN), ',')))

In [ ]:
# 최종 구조로 처음 분할(4절)에서 다시 학습해 1회 테스트 정확도를 잰다
model_final = build(FINAL_HIDDEN)
hist_final = model_final.fit(X_tr, y_tr, epochs=EPOCHS, batch_size=16, verbose=0)
prob_te = proba(model_final, X_te)
te['NN'] = prob_te.argmax(1) + 1
te['CONF'] = prob_te.max(1)
acc_final_train = accuracy_score(tr['LABEL'], proba(model_final, X_tr).argmax(1) + 1)
acc_final = accuracy_score(te['LABEL'], te['NN'])
print('신경망 %s  학습 정확도 %.3f / 테스트 정확도 %.3f' % (FINAL_HIDDEN, acc_final_train, acc_final))

## 9. 오답 분석

(9주차 05 와 같은 방식) 최종 구조 신경망이 4절 테스트 60건에서 틀린 법안을 본다. 분야별 정확도, 혼동행렬, 틀린 법안 목록, 그리고 법률명이 같아도 분야가 갈리는 법률 구간에서 세 방법을 비교한다.

In [ ]:
print(classification_report(te['LABEL'], te['NN'], labels=list(FIELD), target_names=list(FIELD.values()), zero_division=0))

cm = confusion_matrix(te['LABEL'], te['NN'], labels=list(FIELD))
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(9)); ax.set_yticks(range(9))
ax.set_xticklabels(list(FIELD), fontsize=9); ax.set_yticklabels([f'{k} {v}' for k, v in FIELD.items()], fontsize=8)
for i in range(9):
    for j in range(9):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('예측'); ax.set_ylabel('정답'); ax.set_title('혼동행렬 (신경망 %s, 테스트)' % (FINAL_HIDDEN,))
plt.tight_layout(); plt.show()

In [ ]:
wrong = te[te['NN'] != te['LABEL']].copy()
wrong['정답'] = wrong['LABEL'].map(FIELD); wrong['예측'] = wrong['NN'].map(FIELD)
wrong['규칙'] = wrong['RULE'].map(FIELD); wrong['로지스틱'] = wrong['LOGREG'].map(FIELD)
wrong['확신도'] = wrong['CONF'].round(2)
wrong['제안이유(앞부분)'] = wrong['TEXT'].str[:60]
print('신경망 오답', len(wrong), '건 / 그중 애매 표시', (wrong['AMBIG'] == 'Y').sum(), '건 / 확신도 0.6 미만', (wrong['CONF'] < 0.6).sum(), '건')
print('맞힌 법안의 평균 확신도 %.2f / 틀린 법안의 평균 확신도 %.2f' % (te.loc[te.NN == te.LABEL, 'CONF'].mean(), wrong['CONF'].mean()))
pd.set_option('display.max_colwidth', 70)
wrong[['BILL_NAME', '정답', '예측', '확신도', '규칙', '로지스틱', '제안이유(앞부분)']]

In [ ]:
seg = te['LAW'].isin(mixed_laws)
res = pd.DataFrame({
    '구간': ['전체', '분야가 갈리는 법률', '분야가 하나인 법률'],
    '건수': [len(te), seg.sum(), (~seg).sum()],
    **{name: [accuracy_score(te.LABEL, te[c]), accuracy_score(te[seg].LABEL, te[seg][c]), accuracy_score(te[~seg].LABEL, te[~seg][c])]
       for name, c in [('규칙 기반', 'RULE'), ('로지스틱 회귀', 'LOGREG'), ('신경망', 'NN')]}
}).round(3)
ax = res.set_index('구간')[['규칙 기반', '로지스틱 회귀', '신경망']].plot.bar(figsize=(8, 3.5), rot=0, ylim=(0, 1))
ax.set_ylabel('테스트 정확도'); ax.set_title('구간별 비교 (4절 테스트 60건)')
plt.tight_layout(); plt.show()
res

오답을 읽어 보면 대부분 한 법안이 두 분야에 걸친 경우다(예: 공동주택관리법인데 주차 문제, 승강기 안전관리법인데 검사 기관 문제). 글자 묶음은 법률 이름과 자주 나오는 낱말에 끌려가므로, 법률명이 같은데 분야가 갈리는 법률 구간에서 세 방법 모두 정확도가 떨어진다. 틀린 법안은 확신도가 낮은 편이어서, 화면에서는 확신도 0.6 미만에 "분류 확인 필요"를 붙여 사람이 다시 보게 한다(11절).

## 10. 결과표

아래 표는 바로 위 실행 결과로 만든다. 1회 = 4절 분할(테스트 60건), 10회 = 8절 분할 10번의 평균±표준편차.

In [ ]:
def ms(name):
    return '%.3f ± %.3f' % (cv.loc[name, 'mean'], cv.loc[name, 'std'])
final_name = '신경망 ' + str(FINAL_HIDDEN)
result = pd.DataFrame([
    {'모델': '규칙 기반', '테스트 정확도(1회)': round(acc_rule, 3), '10회 평균±표준편차': ms('규칙 기반'), '비고': '법률명 → 가장 많은 분야'},
    {'모델': '로지스틱 회귀', '테스트 정확도(1회)': round(acc_lr, 3), '10회 평균±표준편차': ms('로지스틱 회귀'), '비고': 'TF-IDF 같음, C=%s' % LR_C},
    {'모델': '신경망(최종 구조) %s' % (FINAL_HIDDEN,), '테스트 정확도(1회)': round(acc_final, 3), '10회 평균±표준편차': ms(final_name), '비고': 'Dense ReLU → Softmax 9, 20 epoch'},
])
result

| 모델 | 테스트 정확도(1회) | 10회 평균±표준편차 | 비고 |
|---|---|---|---|
| 규칙 기반 | 0.717 | 0.757 ± 0.069 | 법률명 → 가장 많은 분야 |
| 로지스틱 회귀 | 0.800 | 0.808 ± 0.053 | 같은 TF-IDF, C=100 (훈련 데이터 5겹 교차검증으로 고름) |
| 신경망(최종 구조) (128,) | 0.800 | 0.790 ± 0.047 | Dense 128 ReLU → Softmax 9, 20 epoch |

참고: 9주차 구조 (512, 256) 는 1회 0.733, 10회 0.792 ± 0.053. 은닉층 네 가지의 10회 평균은 0.783~0.793 으로 거의 같아서 가장 작은 (128,) 을 골랐다.

**읽는 법**

- 신경망과 로지스틱 회귀 모두 규칙 기반보다 10회 평균으로 약 0.03~0.05 높다. 법률명만 보는 것보다 제안이유 글을 읽는 쪽이 낫다.
- 신경망과 로지스틱 회귀의 차이(0.790 대 0.808)는 표준편차(약 0.05)보다 훨씬 작아서 "어느 쪽이 낫다"고 말할 수 없다. 라벨이 298건뿐이라 은닉층이 주는 이점이 거의 드러나지 않는다. 라벨을 더 붙이면 다시 비교할 만하다.
- 신경망과 로지스틱 회귀 모두 학습 정확도는 1.0 에 가깝고 테스트는 0.8 안팎이다(7절). 데이터가 적어 과적합이 있고, 조기 종료로 조금(0.733 → 0.750) 나아진다.
- 화면에 쓰는 모델은 이 수업에서 직접 학습한 신경망 (128,) 이다. 정확도 차이가 흔들림 안에 있으므로 수업 목표인 신경망을 쓰고, 출력 확률(확신도)을 "분류 확인 필요" 표시에 쓴다.

## 11. 화면에 쓰이는 곳

최종 구조로 **라벨 298건 전체**를 다시 학습해 두 가지를 만든다.

1. `data/bills_field_pred.csv`: 수집한 법안 전체의 분야 예측 (BILL_NO, BILL_NAME, FIELD_PRED, FIELD_NAME, CONF, FIELD_LABEL). 지도 화면(`web/app.js`)과 관리자 화면(`web/admin.js`)의 **분야 필터**가 이 값을 쓴다. 확신도(CONF)가 **0.6 미만**이면 화면에 **"분류 확인 필요"** 표시가 붙는다. 사람이 붙인 라벨이 있으면 FIELD_LABEL 에 함께 적는다.
2. `web/data/models/field_classifier.json`: 학습한 가중치와 TF-IDF 사전. 서버(`server/lib/nn.js`)가 TensorFlow 없이 같은 계산을 해서, 새로 들어온 법안의 분야를 바로 붙인다(에이전트 도구 `classify_field`). 파일 안의 `check` 예시 5건은 Keras 예측값이고, `node server/test/nn.js` 가 서버 계산과 같은지 확인한다.

In [ ]:
vec_final = make_vec()
X_all = vec_final.fit_transform(df['TEXT']).toarray().astype('float32')
y_all = to_categorical(df['LABEL'] - 1, num_classes=9)
final = build(FINAL_HIDDEN, n_in=X_all.shape[1])
final.fit(X_all, y_all, epochs=EPOCHS, batch_size=16, verbose=0)
print('최종 모델', FINAL_HIDDEN, '/ 학습 법안', len(df), '건 / 입력', X_all.shape[1], '차원')
print('라벨 법안 학습 정확도 %.3f' % accuracy_score(df['LABEL'], proba(final, X_all).argmax(1) + 1))

pred = allb.copy()
pred['TEXT'] = pred['SUMMARY'].fillna('').map(clean)
prob = proba(final, vec_final.transform(pred['TEXT']).toarray().astype('float32'))
pred['FIELD_PRED'] = prob.argmax(1) + 1
pred['FIELD_NAME'] = pred['FIELD_PRED'].map(FIELD)
pred['CONF'] = prob.max(1).round(3)
known = lab.set_index('BILL_NO')['LABEL']
pred['FIELD_LABEL'] = pred['BILL_NO'].map(known)          # 사람이 확인한 라벨이 있으면 함께 기록
out = pred[['BILL_NO', 'BILL_NAME', 'FIELD_PRED', 'FIELD_NAME', 'CONF', 'FIELD_LABEL']]

# 이전 예측(main)과 비교: 몇 건의 분야가 바뀌었나
old = load('bills_field_pred.csv', required=False)
if old is not None:
    cmp = out.merge(old[['BILL_NO', 'FIELD_PRED', 'CONF']], on='BILL_NO', how='left', suffixes=('', '_OLD'))
    both = cmp['FIELD_PRED_OLD'].notna()
    old_nb = cmp['BILL_NO'].isin(old['BILL_NO'].iloc[:678])   # 앞 678줄 = 이전 노트북 예측, 그 뒤 = 매일 수집 때 임시 분류기(확신도 0.5)가 덧붙인 줄
    ch = both & (cmp['FIELD_PRED'] != cmp['FIELD_PRED_OLD'])
    print('이전 예측이 있는 법안 %d건 중 분야가 바뀐 법안 %d건 (노트북 예측이던 %d건 중 %d건, 임시 분류 %d건 중 %d건) / 새 법안 %d건'
          % (both.sum(), ch.sum(), (both & old_nb).sum(), (ch & old_nb).sum(), (both & ~old_nb).sum(), (ch & ~old_nb).sum(), (~both).sum()))

data_dir = '../data' if os.path.isdir('../data') else '.'
out.to_csv(os.path.join(data_dir, 'bills_field_pred.csv'), index=False, encoding='utf-8-sig')
print('저장:', os.path.join(data_dir, 'bills_field_pred.csv'), len(out), '건')
print(out['FIELD_NAME'].value_counts().to_string())
print('확신도 0.6 미만:', (out['CONF'] < 0.6).sum(), '건 → 화면에 "분류 확인 필요"')

In [ ]:
# 서버용 모델 내보내기 (server/lib/nn.js 형식)
def b64(a):
    return base64.b64encode(np.asarray(a, dtype='<f4').tobytes()).decode('ascii')    # float32, 행 = 입력, 열 = 출력

dense = [L for L in final.layers if isinstance(L, Dense)]
layers = []
for L in dense:
    W, bias = L.get_weights()                     # Keras Dense: W 모양 (입력, 출력)
    layers.append({'units': int(W.shape[1]), 'activation': L.get_config()['activation'],
                   'W': b64(W), 'b': [round(float(x), 6) for x in bias]})
assert dense[0].get_weights()[0].shape[0] == len(vec_final.idf_)

pick = df.groupby('LABEL').head(1).head(5)        # 서로 다른 분야 5건
check_in = list(pick['TEXT'])
check_out = proba(final, vec_final.transform(check_in).toarray().astype('float32'))

export = {
    'name': 'field_classifier',
    'input': {'type': 'tfidf_char_wb', 'ngram': [2, 3], 'lowercase': True, 'sublinear_tf': True, 'norm': 'l2',
              'vocab': {g: int(i) for g, i in vec_final.vocabulary_.items()},
              'idf': [round(float(x), 6) for x in vec_final.idf_]},
    'layers': layers,
    'labels': [str(k) for k in FIELD],
    'metrics': {'test_acc': round(float(acc_final), 4), 'cv_mean': round(float(cv.loc[final_name, 'mean']), 4),
                'cv_std': round(float(cv.loc[final_name, 'std']), 4), 'logreg_test_acc': round(float(acc_lr), 4),
                'rule_test_acc': round(float(acc_rule), 4), 'n_train': int(len(df)), 'hidden': list(FINAL_HIDDEN),
                'trained_at': datetime.date.today().isoformat()},
    'check': [{'input': t, 'output': [round(float(p), 6) for p in row]} for t, row in zip(check_in, check_out)],
}
model_dir = '../web/data/models' if os.path.isdir('../web/data') else '.'
os.makedirs(model_dir, exist_ok=True)
path = os.path.join(model_dir, 'field_classifier.json')
with open(path, 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, separators=(',', ':'))
print('저장:', path, '%.2f MB' % (os.path.getsize(path) / 1e6))

In [ ]:
# 내보낸 파일만으로 다시 계산해 Keras 예측과 같은지 확인 (서버가 하는 계산을 numpy 로)
J = json.load(open(path, encoding='utf-8'))
def tfidf_from_json(text, spec):
    cnt = {}
    for w in text.lower().split():
        w = ' ' + w + ' '
        for n in range(spec['ngram'][0], spec['ngram'][1] + 1):
            for i in range(max(1, len(w) - n + 1)):
                c = spec['vocab'].get(w[i:i + n])
                if c is not None: cnt[c] = cnt.get(c, 0) + 1
    x = np.zeros(len(spec['idf']))
    for c, k in cnt.items(): x[c] = (1 + np.log(k)) * spec['idf'][c]
    return x / (np.linalg.norm(x) or 1)
def forward(x):
    n_in = len(x)
    for L in J['layers']:
        W = np.frombuffer(base64.b64decode(L['W']), dtype='<f4').reshape(n_in, L['units'])
        z = x @ W + np.array(L['b'])
        x = np.maximum(z, 0) if L['activation'] == 'relu' else np.exp(z - z.max()) / np.exp(z - z.max()).sum()
        n_in = L['units']
    return x
diff = max(np.abs(forward(tfidf_from_json(c['input'], J['input'])) - np.array(c['output'])).max() for c in J['check'])
print('내보낸 파일로 계산한 값과 Keras 예측의 최대 차이: %.2e' % diff)
assert diff < 1e-3
print('check 예시:', [FIELD[int(np.argmax(c['output'])) + 1] for c in J['check']])

**확신도에 대해**: 최종 구조 (128,) 는 이전 노트북의 (512, 256) 보다 확률을 덜 극단적으로 낸다. 그래서 확신도 0.6 미만("분류 확인 필요")이 이전보다 많다(위 출력 참고). 4절 테스트에서 틀린 법안의 평균 확신도가 맞힌 법안보다 낮았으므로(9절), 이 표시는 사람이 다시 볼 법안을 고르는 데 쓸모가 있다.